In [ ]:
import torch
from transformers import MarianMTModel, MarianTokenizer

device = torch.device("cpu")

translator_model = "Helsinki-NLP/opus-mt-en-ru"
tokenizer = MarianTokenizer.from_pretrained(translator_model)
translator = MarianMTModel.from_pretrained(translator_model)

examples = [
    "I love cats and dogs",
    "Hello, how are you?",
    "The weather is nice today"
]

for example in examples:
    tokens = tokenizer(example, return_tensors="pt").to(device)
    translated = translator.generate(**tokens)
    result = tokenizer.decode(translated[0], skip_special_tokens=True)
    print(result)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/803k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/1.08M [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.60M [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/307M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/307M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

Я люблю кошек и собак.
Привет, как дела?
Сегодня хорошая погода.


## Seq2Seq translation model

In [ ]:
from datasets import load_dataset
ds = load_dataset("Helsinki-NLP/opus-100", "en-ru")
print(ds["train"][0])

README.md:   0%|          | 0.00/65.4k [00:00<?, ?B/s]

en-ru/test-00000-of-00001.parquet:   0%|          | 0.00/310k [00:00<?, ?B/s]

en-ru/train-00000-of-00001.parquet:   0%|          | 0.00/124M [00:00<?, ?B/s]

en-ru/validation-00000-of-00001.parquet:   0%|          | 0.00/310k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/1000000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

{'translation': {'en': "Yeah, that's not exactly...", 'ru': 'Да, но не совсем...'}}


In [ ]:
print(ds)

DatasetDict({
    test: Dataset({
        features: ['translation'],
        num_rows: 2000
    })
    train: Dataset({
        features: ['translation'],
        num_rows: 1000000
    })
    validation: Dataset({
        features: ['translation'],
        num_rows: 2000
    })
})


In [ ]:
train_ds = ds['train'].select(range(50000))

In [ ]:
import string
def preprocess(sentence):
  sentence = sentence.lower()
  cleaner = str.maketrans('', '', string.punctuation)
  sentence = sentence.translate(cleaner)
  sentence = sentence.split(' ')
  tokens = ["<SOS>"] + sentence + ["<EOS>"]

  return tokens



In [ ]:
print(preprocess("Hello, I love my cat - Mushi very much!"))

['<SOS>', 'hello', 'i', 'love', 'my', 'cat', '', 'mushi', 'very', 'much', '<EOS>']


In [ ]:
eng_sentences = []
ru_sentences = []
for line in train_ds:
  eng_sentences.append(preprocess(line['translation']['en']))
  ru_sentences.append(preprocess(line['translation']['ru']))

In [ ]:
class Vocabulary:
  def __init__(self):
    self.word2index = {'<PAD>': 0, '<SOS>': 1, '<EOS>': 2, '<UNK>': 3}
    self.index2word = {0: '<PAD>', 1: '<SOS>', 2: '<EOS>', 3: '<UNK>'}
    self.n_words = 4

  def add_sentence(self, sentence):
    for word in sentence:
      self.add_word(word)

  def add_word(self, word):
    if word not in self.word2index:
      self.word2index[word] = self.n_words
      self.index2word[self.n_words] = word
      self.n_words += 1

In [ ]:
eng_vocab = Vocabulary()
for s in eng_sentences:
  eng_vocab.add_sentence(s)
ru_vocab = Vocabulary()
for s in ru_sentences:
  ru_vocab.add_sentence(s)

In [ ]:
print(eng_vocab.n_words)
print(ru_vocab.n_words)
# russian is morphologicaly richer than english

40285
76395


In [ ]:
def sentence_to_indices(sentence, vocab):
  indices = []
  for word in sentence:
    word = vocab.word2index[word]
    indices.append(word)
  return indices

print(sentence_to_indices(eng_sentences[0], eng_vocab))

[1, 4, 5, 6, 7, 2]


In [ ]:
eng_indices = []
ru_indices = []
for eng_sentence in eng_sentences:
  eng_indices.append(sentence_to_indices(eng_sentence,eng_vocab))
for ru_sentence in ru_sentences:
  ru_indices.append(sentence_to_indices(ru_sentence,ru_vocab))

In [ ]:
print(eng_sentences[0])
print(ru_sentences[0])

['<SOS>', 'yeah', 'thats', 'not', 'exactly', '<EOS>']
['<SOS>', 'да', 'но', 'не', 'совсем', '<EOS>']
